# Accumulator: voltage, current, state of charge, temperature

The pack as an equivalent circuit (`lapsim/subsystems/power/battery.py`): open-circuit voltage from the
chemistry's OCV curve, ohmic resistance plus two RC branches (with an Arrhenius temperature factor), one
lumped thermal node per cell, and the current limits that derate with temperature. The endurance is run lap
by lap with the state of charge and temperatures carried over, and a driver change at half distance.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
CAR = lapsim.load_car("ct17_ev")
SL = [slider(CAR, "pack:pack.series", 100, 160, 1, "cells in series"),
      slider(CAR, "pack:pack.parallel", 2, 6, 1, "cells in parallel"),
      slider(CAR, "pack:cell.r0", 5, 30, 0.5, "cell R0", 1e3, "mOhm"),
      slider(CAR, "pack:cell.capacity_ah", 2.5, 5.5, 0.1, "cell capacity", 1, "Ah"),
      slider(CAR, "pack:thermal.r_surface_ambient", 2, 30, 0.5, "cell cooling resistance", 1, "K/W"),
      slider(CAR, "ev:strategy.endurance_power_limit", 20, 80, 1, "endurance power limit", 1e-3, "kW")]
SL[-1].value = 80
out = W.Output()


def run():
    c = with_values(CAR, SL)
    res = c.run(("endurance",))
    display(res.endurance.round(3).iloc[[0, len(res.endurance) // 2, -1]])
    laps = res.laps["endurance_all"]
    t0, parts = 0.0, []
    for lr in laps:
        f = lr.frame()
        f["t_total"] = t0 + f["t"]
        t0 += lr.time
        parts.append(f)
    d = pd.concat(parts)
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
    for a, (col, lab) in zip(ax, (("soc", "state of charge"), ("V_pack", "pack voltage [V]"),
                                  ("I_pack", "pack current [A]"), ("cell_temp_C", "cell temperature [degC]"))):
        a.plot(d["t_total"] / 60, d[col], lw=0.6); a.set(xlabel="time [min]", title=lab)
    plt.show()
    for n in res.notes:
        print("note:", n)


display(W.VBox([*SL, run_button("Run the endurance", run, out), out]))
with out:
    run()